<a href="https://colab.research.google.com/github/ArthurrCr/cloudband/blob/main/notebooks/00_baselines/score_ocm_cloudsen12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --quiet "tacoreader<1.0" omnicloudmask==1.7.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 15.8 MB/s eta 0:00:00


In [2]:
REPO_URL = "https://github.com/ArthurrCr/cloudband.git"
PROJECT_DIR = "/content/cloudband"
BRANCH = "main"

import os
import sys

if not os.path.exists(PROJECT_DIR):
    !git clone --quiet {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git fetch --quiet origin && git reset --quiet --hard origin/{BRANCH}

SRC_DIR = f"{PROJECT_DIR}/src"
os.chdir(PROJECT_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

!PYTHONPATH={SRC_DIR} python -m pytest tests -q

........................................................................ [ 74%]
.........................                                                [100%]
=============================== warnings summary ===============================
tests/contract/test_pipeline.py::test_wrong_raster_size_is_rejected
tests/contract/test_pipeline.py::test_duplicate_predictions_are_rejected
tests/contract/test_pipeline.py::test_attach_and_score_round_trip
tests/contract/test_pipeline.py::test_pooled_counts_equal_whole_collection
  /usr/local/lib/python3.13/dist-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
    dataset = writer(

tests/contract/test_pipeline.py::test_wrong_raster_size_is_rejected
tests/contract/test_pipeline.py::test_attach_and_score_round_trip
tests/contract/test_pipeline.py::test_pooled_counts_equal_whole_collection
  /usr/local/lib/python3.13/dist-packages/rasterio/__init__.py:367: No

In [3]:
from pathlib import Path

from cloudband.baselines import ocm
from cloudband.colab.session import reload_package, start
from cloudband.datasets import cloudsen12 as ds
from cloudband.pipelines import phase0

reload_package("cloudband")

In [4]:
def report(position, total):
    if position % 25 == 0 or position == total:
        print(f"{position}/{total}", flush=True)

In [5]:
from google.colab import drive

drive.mount("/content/drive")

RESULTS_DIR = Path("/content/drive/MyDrive/cloudband/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

session = start(PROJECT_DIR, require_accelerator=True)
print(f"results: {RESULTS_DIR}")

Mounted at /content/drive
project: /content/cloudband
device: cuda:Tesla T4
free disk: 65.4 GiB
omnicloudmask: 1.7.0
rasterio: 1.5.1
pandas: 2.2.3
numpy: 2.1.3
results: /content/drive/MyDrive/cloudband/results


In [6]:
table = phase0.load_test_split()
print(f"scenes: {len(table)}")
print("pairable:", ds.expected_pairable_scenes(table))

scenes: 975
pairable: {'clear': 705, 'cloud': 767, 'shadow': 655}


In [7]:
LIMIT = None

ocm.check_version()
config = ocm.InferenceConfig(
    patch_size=509,
    patch_overlap=0,
    model_version=ocm.LATEST_MODEL_VERSION,
)
print(config)

InferenceConfig(patch_size=509, patch_overlap=0, inference_dtype='float32', batch_size=1, model_version=4.0, inference_device=None, extra={})


In [8]:
result = phase0.run(
    table,
    lambda stack: ocm.predict_array(stack, config),
    model_id=f"ocm-rgn-published-v{ocm.package_version()}",
    limit=LIMIT,
    progress=report,
)
result.scores

PM_model_OCM_7.97_R_G_NIR_3_smp_regnety_(…): reconstructing file:   0%|          |  0.00B / 27.1MB            

PM_model_OCM_7.97_R_G_NIR_3_smp_regnety_(…): downloading bytes:           |  0.00B            

PM_model_OCM_7.97_R_G_NIR_3_smp_edgenext(…): reconstructing file:   0%|          |  0.00B / 30.7MB            

PM_model_OCM_7.97_R_G_NIR_3_smp_edgenext(…): downloading bytes:           |  0.00B            

25/975
50/975
75/975
100/975
125/975
150/975
175/975
200/975
225/975
250/975
275/975
300/975
325/975
350/975
375/975
400/975
425/975
450/975
475/975
500/975
525/975
550/975
575/975
600/975
625/975
650/975
675/975
700/975
725/975
750/975
775/975
800/975
825/975
850/975
875/975
900/975
925/975
950/975
975/975


,tp,tn,fp,fn,ua,pa,oa,boa,f1,iou
experiment,,,,,,,,,,
clear,127851868,110577526,8067248,6107333,94.06,95.44,94.39,94.32,94.75,90.02
cloud,87421283,150704234,7383521,7094937,92.21,92.49,94.27,93.91,92.35,85.79
shadow,18682841,225278207,3197214,5445713,85.39,77.43,96.58,88.02,81.21,68.37


In [9]:
frame = phase0.compare_to_reference(result, phase0.STUPMASK_CLOUDSEN12)
print("pairable scenes:", result.pairable)
print(phase0.describe_comparison(frame))
frame

pairable scenes: {'clear': 776, 'cloud': 776, 'shadow': 671}
fidelity check against STUPmask paper, section 4.4.1, measured on cloudsen12plus_test_p509_high


,observed,reference,difference,same_dataset
cloud,93.91,93.64,0.27,True


In [10]:
paths = phase0.save(
    result,
    RESULTS_DIR,
    config=config.as_kwargs(),
    package_versions=session.package_versions,
)
for name, path in paths.items():
    print(f"{name}: {path}")

scores: /content/drive/MyDrive/cloudband/results/ocm-rgn-published-v1.7.0/cloudsen12plus_test_p509_high/scores.csv
confusion: /content/drive/MyDrive/cloudband/results/ocm-rgn-published-v1.7.0/cloudsen12plus_test_p509_high/confusion.csv
per_scene_boa: /content/drive/MyDrive/cloudband/results/ocm-rgn-published-v1.7.0/cloudsen12plus_test_p509_high/per_scene_boa.csv
manifest: /content/drive/MyDrive/cloudband/results/ocm-rgn-published-v1.7.0/cloudsen12plus_test_p509_high/manifest.json
